# 04 · Join trips to taxi zones

Combine a Parquet fact table with a CSV dimension and prove the join does not multiply or discard trips.

Open this notebook in Google Colab with a CPU runtime, or use a Linux x86-64
Jupyter environment with Python 3.11 or newer. No database account is required.
Run the cells in order. The first cell downloads a verified Kelvo preview binary
and installs pinned Python readers; it does not compile Kelvo.

Every analytical SQL query below runs through the **Kelvo executable**. PyArrow
reads its Arrow output, and pandas/matplotlib format small results. The pinned
DuckDB driver materializes execution before delivering Arrow batches; delivery
backpressure is not a worker memory bound.


**Data:** [NYC TLC trip records and taxi-zone lookup](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).
This lesson uses the January 2024 yellow-taxi Parquet file. See the
[NYC Open Data terms of use](https://opendata.cityofnewyork.us/overview/#termsofuse)
and TLC's data dictionary and accuracy notice; do not treat public availability
as a warranty or apply the notebook's code license to the data.
The original Parquet download is roughly 50 MB and contains roughly three million
rows. The helper checks the pinned file digest. Queries use explicit result
limits; a small result does not imply a small source scan.


[All notebooks](https://github.com/SYNEHQ/kelvo-go/blob/cargo/notebooks/README.md)
| [Kelvo documentation](https://github.com/SYNEHQ/kelvo-go/blob/cargo/docs/README.md)

## Set up a private lab


In [ ]:
# Default setup installs pinned Python packages and verifies the reviewed helper.
# Operators validating a checkout can explicitly set KELVO_NOTEBOOK_SKIP_INSTALL=1
# and KELVO_NOTEBOOK_HELPER=/absolute/path/to/kelvo_notebooks.py before starting.
import hashlib
import importlib.util
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import urllib.parse
import urllib.request

KELVO_NOTEBOOK_REF = "eacb7e2aedd8020575ecfede53db819112c40199"
KELVO_HELPER_SHA256 = "86696a66a3c903eb8f18d529fcef055e8bee72234da324b8776529138436d9a9"
if os.environ.get("KELVO_NOTEBOOK_SKIP_INSTALL") != "1":
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet",
        "pyarrow==25.0.1", "pandas==2.3.3", "matplotlib==3.10.7", "PyYAML==6.0.3",
    ], check=True)

helper_directory = None
local_helper = os.environ.get("KELVO_NOTEBOOK_HELPER")
if local_helper:
    helper_path = Path(local_helper).expanduser().resolve(strict=True)
else:
    helper_url = (
        "https://raw.githubusercontent.com/SYNEHQ/kelvo-go/"
        + KELVO_NOTEBOOK_REF + "/notebooks/kelvo_notebooks.py"
    )
    request = urllib.request.Request(helper_url, headers={"User-Agent": "kelvo-notebooks"})
    with urllib.request.urlopen(request, timeout=60) as response:
        final_url = response.geturl()
        if urllib.parse.urlsplit(final_url).scheme != "https" or final_url != helper_url:
            raise ValueError("Helper download did not retain its pinned HTTPS URL")
        payload = response.read(1024 * 1024 + 1)
    if len(payload) > 1024 * 1024:
        raise ValueError("Unexpected helper size")
    if hashlib.sha256(payload).hexdigest() != KELVO_HELPER_SHA256:
        raise ValueError("Helper checksum mismatch")
    helper_directory = Path(tempfile.mkdtemp(prefix="kelvo-notebook-helper-"))
    helper_path = helper_directory / "kelvo_notebooks.py"
    helper_path.write_bytes(payload)

spec = importlib.util.spec_from_file_location("kelvo_notebooks", helper_path)
helper = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helper
spec.loader.exec_module(helper)
from kelvo_notebooks import Lab, remote_query

lab = Lab.create()
print("Kelvo workspace:", lab.root)


In [ ]:
taxi, zones = lab.taxi(), lab.zones()
config = lab.catalog(sources=[
    {"id": "trips", "type": "parquet", "path": str(taxi)},
    {"id": "zones", "type": "csv", "path": str(zones)},
])
keys = lab.query(config, """
    SELECT COUNT(*)::BIGINT AS rows,
           COUNT(DISTINCT "LocationID")::BIGINT AS distinct_keys,
           COUNT(*) FILTER (WHERE "LocationID" IS NULL)::BIGINT AS missing_keys
    FROM zones
""", sources=["zones"]).table().to_pylist()[0]
assert keys["rows"] == keys["distinct_keys"] > 0
assert keys["missing_keys"] == 0
print("Dimension key contract:", keys)


## Keep unmatched trips visible

A left join retains fact rows. Group missing lookups into a visible bucket so
you can investigate them rather than silently losing them.


In [ ]:
boroughs = lab.query(config, """
    SELECT COALESCE(z."Borough", 'Unmapped') AS borough,
           COUNT(*)::BIGINT AS trips,
           COUNT(*) FILTER (WHERE z."LocationID" IS NULL)::BIGINT AS unmatched
    FROM trips t LEFT JOIN zones z ON t."PULocationID" = z."LocationID"
    WHERE t.tpep_pickup_datetime >= TIMESTAMP '2024-01-01'
      AND t.tpep_pickup_datetime < TIMESTAMP '2024-01-08'
    GROUP BY 1 ORDER BY trips DESC, borough
""", sources=["trips", "zones"]).table()
expected = lab.query(config, """
    SELECT COUNT(*)::BIGINT AS trips FROM trips
    WHERE tpep_pickup_datetime >= TIMESTAMP '2024-01-01'
      AND tpep_pickup_datetime < TIMESTAMP '2024-01-08'
""", sources=["trips"]).table()["trips"][0].as_py()
assert sum(boroughs["trips"].to_pylist()) == expected
assert sum(boroughs["unmatched"].to_pylist()) <= expected
display(boroughs.to_pandas())


## Enrich both sides of a trip

The two aliases below refer to one registered dimension, so its unique-key
contract protects both joins.


In [ ]:
routes = lab.query(config, """
    SELECT COALESCE(p."Zone", 'Unmapped') AS pickup_zone,
           COALESCE(d."Zone", 'Unmapped') AS dropoff_zone,
           COUNT(*)::BIGINT AS trips
    FROM trips t
    LEFT JOIN zones p ON t."PULocationID" = p."LocationID"
    LEFT JOIN zones d ON t."DOLocationID" = d."LocationID"
    WHERE t.tpep_pickup_datetime >= TIMESTAMP '2024-01-01'
      AND t.tpep_pickup_datetime < TIMESTAMP '2024-01-08'
    GROUP BY 1, 2 ORDER BY trips DESC, pickup_zone, dropoff_zone LIMIT 15
""", sources=["trips", "zones"], max_rows=15).table()
assert routes.num_rows == 15
assert sum(routes["trips"].to_pylist()) <= expected
display(routes.to_pandas())


## Clean up

Run this cell after inspecting or saving your results. Re-run the setup and data
cells if you want to restart the lesson after cleanup.


In [ ]:
# Close local processes and delete this lesson's private temporary workspace.
# Downloaded release/data caches, if configured by the helper, remain reusable.
lab.close()
if helper_directory is not None:
    shutil.rmtree(helper_directory, ignore_errors=True)
